# SC-4-Errors-Events - Erreurs et Evenements

**Navigation** : [Index](../README.md) | [<< Inheritance](SC-3-Inheritance.ipynb) | [Token Standards >>](../02-Solidity-Advanced/SC-5-Token-Standards.ipynb)

---

## Objectifs d'apprentissage

1. Gerer les erreurs avec `require`, `revert`, `assert`
2. Creer des **custom errors** (Solidity 0.8.4+)
3. Emettre des **events** pour le logging

### Prerequis

- [SC-1](SC-1-Solidity-Basics.ipynb) a [SC-3](SC-3-Inheritance.ipynb) completes
- Comprendre les transactions Ethereum (gas, reverts)

### Duree estimee : 30 minutes

---

## 0. Connexion a la blockchain locale

Tous les contrats de ce notebook sont **compiles et deployes reellement** sur anvil.
Lancez `anvil` dans un terminal avant d'executer les cellules.


In [ ]:
# Connection a anvil (blockchain locale Foundry)
# Prerequis: anvil en cours d'execution dans un terminal
from web3 import Web3
import solcx

SOLC_VERSION = "0.8.28"
ANVIL_URL = "http://127.0.0.1:8545"

# Connexion
w3 = Web3(Web3.HTTPProvider(ANVIL_URL))
assert w3.is_connected(), f"Impossible de se connecter a {ANVIL_URL}. Lancez 'anvil' dans un terminal."

# Installer solc si necessaire
installed = [str(v) for v in solcx.get_installed_solc_versions()]
if SOLC_VERSION not in installed:
    solcx.install_solc(SOLC_VERSION)
solcx.set_solc_version(SOLC_VERSION)

deployer = w3.eth.accounts[0]
print(f"Connecte a anvil (chain {w3.eth.chain_id}), deployer: {deployer[:10]}...")


def compile_and_deploy(w3, source_code, deployer, *constructor_args):
    """Compiler et deployer un contrat Solidity."""
    compiled = solcx.compile_source(
        source_code, output_values=["abi", "bin"], solc_version=SOLC_VERSION
    )
    contract_id, contract_interface = compiled.popitem()
    Contract = w3.eth.contract(
        abi=contract_interface["abi"], bytecode=contract_interface["bin"]
    )
    tx_hash = Contract.constructor(*constructor_args).transact({"from": deployer})
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    instance = w3.eth.contract(
        address=receipt.contractAddress, abi=contract_interface["abi"]
    )
    print(f"Deploye: {contract_id.split(':')[-1]} a {receipt.contractAddress}")
    return instance, receipt


---

## 1. Gestion des erreurs

| Fonction | Usage | Gas remboursé |
|----------|-------|---------------|
| `require` | Conditions d'entree | Oui |
| `revert` | Erreurs complexes | Oui |
| `assert` | Invariants internes | Non |

In [1]:
# Gestion des erreurs
ERROR_HANDLING = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

contract ErrorHandling {
    address public owner;
    uint256 public value;

    constructor() {
        owner = msg.sender;
    }

    // REQUIRE : conditions d'entree (rembourse gas restant)
    function setValue(uint256 _value) public {
        require(msg.sender == owner, "Not owner");
        require(_value > 0, "Value must be positive");
        value = _value;
    }

    // REVERT : erreurs avec logique complexe
    function complexCheck(uint256 a, uint256 b) public pure {
        if (a == 0) {
            revert("A cannot be zero");
        }
        if (b == 0) {
            revert("B cannot be zero");
        }
        if (a + b > 100) {
            revert("Sum too large");
        }
    }

    // ASSERT : invariants (ne rembourse pas le gas)
    // Utilise uniquement pour les bugs internes
    function invariant() public view {
        assert(owner != address(0));  // Ne devrait jamais echouer
    }
}
'''


# Compilation et deploiement reel sur anvil
errorhandling, receipt = compile_and_deploy(w3, ERROR_HANDLING, deployer)

Gestion des erreurs:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

contract ErrorHandling {
    address public owner;
    uint256 public value;

    constructor() {
        owner = msg.sender;
    }

    // REQUIRE : conditions d'entree (rembourse gas restant)
    function setValue(uint256 _value) public {
        require(msg.sender == owner, "Not owner");
        require(_value > 0, "Value must be positive");
        value = _value;
    }

    // REVERT : erreurs avec logique complexe
    function complexCheck(uint256 a, uint256 b) public pure {
        if (a == 0) {
            revert("A cannot be zero");
        }
        if (b == 0) {
            revert("B cannot be zero");
        }
        if (a + b > 100) {
            revert("Sum too large");
        }
    }

    // ASSERT : invariants (ne rembourse pas le gas)
    // Utilise uniquement pour les bugs internes
    function invariant() public view {
        assert(owner != address(0));  // Ne devrait jamais echouer
 

---

## 2. Custom Errors (Solidity 0.8.4+)

Les custom errors sont plus economiques en gas que les strings.

In [2]:
# Custom errors
CUSTOM_ERRORS = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

// Definition des erreurs custom (hors contrat)
error InsufficientBalance(uint256 available, uint256 required);
error Unauthorized(address caller);
error InvalidAmount();

contract CustomErrors {
    mapping(address => uint256) public balances;

    function deposit() public payable {
        balances[msg.sender] += msg.value;
    }

    function withdraw(uint256 amount) public {
        // Custom error avec parametres
        if (balances[msg.sender] < amount) {
            revert InsufficientBalance(balances[msg.sender], amount);
        }
        
        balances[msg.sender] -= amount;
        payable(msg.sender).transfer(amount);
    }

    function transfer(address to, uint256 amount) public {
        if (amount == 0) {
            revert InvalidAmount();
        }
        if (balances[msg.sender] < amount) {
            revert InsufficientBalance(balances[msg.sender], amount);
        }
        
        balances[msg.sender] -= amount;
        balances[to] += amount;
    }
}

// Comparaison gas:
// require(balances[msg.sender] >= amount, "Insufficient balance");
// vs
// revert InsufficientBalance(balances[msg.sender], amount);
// Custom error: ~50% moins couteux
'''


# Compilation et deploiement reel sur anvil
customerrors, receipt = compile_and_deploy(w3, CUSTOM_ERRORS, deployer)

Custom errors (economie de gas):

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

// Definition des erreurs custom (hors contrat)
error InsufficientBalance(uint256 available, uint256 required);
error Unauthorized(address caller);
error InvalidAmount();

contract CustomErrors {
    mapping(address => uint256) public balances;

    function deposit() public payable {
        balances[msg.sender] += msg.value;
    }

    function withdraw(uint256 amount) public {
        // Custom error avec parametres
        if (balances[msg.sender] < amount) {
            revert InsufficientBalance(balances[msg.sender], amount);
        }

        balances[msg.sender] -= amount;
        payable(msg.sender).transfer(amount);
    }

    function transfer(address to, uint256 amount) public {
        if (amount == 0) {
            revert InvalidAmount();
        }
        if (balances[msg.sender] < amount) {
            revert InsufficientBalance(balances[msg.sender], amount);
        }

        

---

## 3. Events

Les events permettent de logger des donnees sur la blockchain.

In [3]:
# Events
EVENTS_EXAMPLE = '''
contract EventsExample {
    // Declaration des events
    event Deposit(address indexed from, uint256 amount, uint256 timestamp);
    event Withdrawal(address indexed to, uint256 amount);
    event Transfer(address indexed from, address indexed to, uint256 amount);
    event OwnershipTransferred(address indexed previousOwner, address indexed newOwner);

    address public owner;
    mapping(address => uint256) public balances;

    constructor() {
        owner = msg.sender;
    }

    function deposit() public payable {
        balances[msg.sender] += msg.value;
        
        // Emission de l'event
        emit Deposit(msg.sender, msg.value, block.timestamp);
    }

    function withdraw(uint256 amount) public {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        
        balances[msg.sender] -= amount;
        payable(msg.sender).transfer(amount);
        
        emit Withdrawal(msg.sender, amount);
    }

    function transferFunds(address to, uint256 amount) public {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        
        balances[msg.sender] -= amount;
        balances[to] += amount;
        
        emit Transfer(msg.sender, to, amount);
    }

    function transferOwnership(address newOwner) public {
        require(msg.sender == owner, "Not owner");
        require(newOwner != address(0), "Invalid address");
        
        address previousOwner = owner;
        owner = newOwner;
        
        emit OwnershipTransferred(previousOwner, newOwner);
    }
}
'''

print("Events pour logging:")
print(EVENTS_EXAMPLE)

Events pour logging:

contract EventsExample {
    // Declaration des events
    event Deposit(address indexed from, uint256 amount, uint256 timestamp);
    event Withdrawal(address indexed to, uint256 amount);
    event Transfer(address indexed from, address indexed to, uint256 amount);
    event OwnershipTransferred(address indexed previousOwner, address indexed newOwner);

    address public owner;
    mapping(address => uint256) public balances;

    constructor() {
        owner = msg.sender;
    }

    function deposit() public payable {
        balances[msg.sender] += msg.value;

        // Emission de l'event
        emit Deposit(msg.sender, msg.value, block.timestamp);
    }

    function withdraw(uint256 amount) public {
        require(balances[msg.sender] >= amount, "Insufficient balance");

        balances[msg.sender] -= amount;
        payable(msg.sender).transfer(amount);

        emit Withdrawal(msg.sender, amount);
    }

    function transferFunds(address to, uint256

### 3.1 Indexed parameters

- Maximum **3 parametres indexes** par event
- Les parametres indexes sont recherchables dans les logs
- Les parametres non indexes sont stockes dans `data`

In [4]:
# Indexed parameters
INDEXED_EXAMPLE = '''
contract IndexedExample {
    // indexed = recherchable par le frontend
    event OrderCreated(
        uint256 indexed orderId,      // Indexe: peut filtrer par orderId
        address indexed buyer,        // Indexe: peut filtrer par buyer
        address indexed seller,       // Indexe: peut filtrer par seller
        uint256 amount,               // Non indexe: stocke dans data
        string productId              // Non indexe: stocke dans data
    );

    function createOrder(
        address seller,
        uint256 amount,
        string memory productId
    ) public returns (uint256) {
        uint256 orderId = uint256(keccak256(abi.encodePacked(
            msg.sender, seller, amount, block.timestamp
        )));
        
        emit OrderCreated(orderId, msg.sender, seller, amount, productId);
        
        return orderId;
    }
}

// Filtrage cote frontend:
// contract.events.OrderCreated({filter: {buyer: userAddress}})
// contract.events.OrderCreated({filter: {orderId: 123}})
'''

print("Indexed parameters pour recherche:")
print(INDEXED_EXAMPLE)

Indexed parameters pour recherche:

contract IndexedExample {
    // indexed = recherchable par le frontend
    event OrderCreated(
        uint256 indexed orderId,      // Indexe: peut filtrer par orderId
        address indexed buyer,        // Indexe: peut filtrer par buyer
        address indexed seller,       // Indexe: peut filtrer par seller
        uint256 amount,               // Non indexe: stocke dans data
        string productId              // Non indexe: stocke dans data
    );

    function createOrder(
        address seller,
        uint256 amount,
        string memory productId
    ) public returns (uint256) {
        uint256 orderId = uint256(keccak256(abi.encodePacked(
            msg.sender, seller, amount, block.timestamp
        )));

        emit OrderCreated(orderId, msg.sender, seller, amount, productId);

        return orderId;
    }
}

// Filtrage cote frontend:
// contract.events.OrderCreated({filter: {buyer: userAddress}})
// contract.events.OrderCreate

---

## 4. Try/Catch pour appels externes

In [5]:
# Try/Catch
TRY_CATCH_EXAMPLE = '''
interface IExternalContract {
    function riskyOperation() external returns (bool);
}

contract TryCatchExample {
    event OperationSuccess();
    event OperationFailed(string reason);
    event OperationFailedBytes(bytes data);

    IExternalContract public externalContract;

    constructor(address _contract) {
        externalContract = IExternalContract(_contract);
    }

    function safeOperation() public returns (bool) {
        // Try/catch pour appels externes uniquement
        try externalContract.riskyOperation() returns (bool success) {
            emit OperationSuccess();
            return success;
        } catch Error(string memory reason) {
            // revert() ou require() avec message
            emit OperationFailed(reason);
            return false;
        } catch Panic(uint256 errorCode) {
            // Erreur assert() ou overflow
            emit OperationFailed(string(abi.encodePacked("Panic: ", errorCode)));
            return false;
        } catch (bytes memory lowLevelData) {
            // Custom error sans message
            emit OperationFailedBytes(lowLevelData);
            return false;
        }
    }
}
'''

print("Try/Catch pour appels externes:")
print(TRY_CATCH_EXAMPLE)

Try/Catch pour appels externes:

interface IExternalContract {
    function riskyOperation() external returns (bool);
}

contract TryCatchExample {
    event OperationSuccess();
    event OperationFailed(string reason);
    event OperationFailedBytes(bytes data);

    IExternalContract public externalContract;

    constructor(address _contract) {
        externalContract = IExternalContract(_contract);
    }

    function safeOperation() public returns (bool) {
        // Try/catch pour appels externes uniquement
        try externalContract.riskyOperation() returns (bool success) {
            emit OperationSuccess();
            return success;
        } catch Error(string memory reason) {
            // revert() ou require() avec message
            emit OperationFailed(reason);
            return false;
        } catch Panic(uint256 errorCode) {
            // Erreur assert() ou overflow
            emit OperationFailed(string(abi.encodePacked("Panic: ", errorCode)));
            

---

## 5. Exercices

### Exercice 1 : Contrat avec custom errors

Creez un contrat de vote avec custom errors pour les cas invalides.

In [ ]:
# Exercice 1 : Contrat de vote avec custom errors
EXERCICE_VOTING = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

// Custom errors
error AlreadyVoted(address voter);
error VotingClosed();
error InvalidProposal(uint256 proposalId);

contract Voting {
    struct Proposal {
        string name;
        uint256 voteCount;
    }

    Proposal[] public proposals;
    mapping(address => bool) public hasVoted;
    bool public votingOpen = true;

    event Voted(address indexed voter, uint256 indexed proposalId);
    event VotingClosed(uint256 timestamp);

    constructor(string[] memory _proposalNames) {
        for (uint i = 0; i < _proposalNames.length; i++) {
            proposals.push(Proposal(_proposalNames[i], 0));
        }
    }

    function vote(uint256 proposalId) public {
        // TODO: Verifier que le vote est ouvert (sinon revert VotingClosed)
        // TODO: Verifier que l utilisateur n a pas deja vote (sinon revert AlreadyVoted)
        // TODO: Verifier que le proposalId est valide (sinon revert InvalidProposal)
        // TODO: Marquer l utilisateur comme ayant vote
        // TODO: Incrementer le compteur de votes de la proposition
        // TODO: Emettre l evenement Voted
    }

    function closeVoting() public {
        // TODO: Fermer le vote
        // TODO: Emettre l evenement VotingClosed avec le timestamp
    }
}
'''

# Deployer (ajuster les arguments du constructeur si necessaire)
# voting, receipt = compile_and_deploy(w3, EXERCICE_VOTING, deployer)
# print(f"Deploye a: {voting.address}")

### Exercice 2 : Contrat Auction avec events

Creez un contrat d'enchere avec events pour chaque action.

In [ ]:
# Exercice 2 : Contrat Auction avec events
EXERCICE_AUCTION = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

error AuctionEnded();
error BidTooLow(uint256 currentBid, uint256 minBid);

contract Auction {
    address public beneficiary;
    uint256 public auctionEndTime;
    address public highestBidder;
    uint256 public highestBid;
    bool public ended;

    mapping(address => uint256) public pendingReturns;

    event HighestBidIncreased(address indexed bidder, uint256 amount);
    event AuctionFinalized(address winner, uint256 amount);
    event BidRefunded(address indexed bidder, uint256 amount);

    constructor(uint256 biddingTime, address _beneficiary) {
        beneficiary = _beneficiary;
        auctionEndTime = block.timestamp + biddingTime;
    }

    function bid() public payable {
        // TODO: Verifier que l enchere n est pas terminee (sinon revert AuctionEnded)
        // TODO: Verifier que l offre est superieure a l offre actuelle (sinon revert BidTooLow)
        // TODO: Rembourser l ancien meilleur encherisseur via pendingReturns
        // TODO: Mettre a jour highestBidder et highestBid
        // TODO: Emettre HighestBidIncreased
    }

    function withdraw() public returns (bool) {
        // TODO: Recuperer le montant en attente pour msg.sender
        // TODO: Si montant > 0, remettre a zero et transferer
        // TODO: Emettre BidRefunded
    }

    function auctionEnd() public {
        // TODO: Verifier que le temps est ecoule
        // TODO: Verifier que l enchere n est pas deja finalisee
        // TODO: Marquer comme finalisee
        // TODO: Emettre AuctionFinalized
        // TODO: Transferer les fonds au beneficiaire
    }
}
'''

# Deployer (ajuster les arguments du constructeur si necessaire)
# auction, receipt = compile_and_deploy(w3, EXERCICE_AUCTION, deployer)
# print(f"Deploye a: {auction.address}")

---

## 6. Resume

| Concept | Description | Gas |
|---------|-------------|-----|
| `require` | Conditions d'entree | Rembourse |
| `revert` | Erreurs complexes | Rembourse |
| `assert` | Invariants internes | Non rembourse |
| `error` | Custom errors (0.8.4+) | Economique |
| `event` | Logging blockchain | - |
| `indexed` | Parametres recherchables | Max 3 |

---

**Notebook suivant** : [SC-5-Token-Standards](../02-Solidity-Advanced/SC-5-Token-Standards.ipynb)